## 04．optunaによる最適化

XGBoostの木の深さや学習率など、最も効率よくWVPを予測できるハイパーパラメータの組み合わせを自動で50回探索させ、ベストなハイパーパラメータを導き出す

②experiment_idでくくる

In [2]:
import numpy as np
import pandas as pd
import xgboost as xgb
import optuna
from sklearn.metrics import mean_squared_error

optuna.logging.set_verbosity(optuna.logging.WARNING)

def optimize_hyperparameters(X_train, X_test, y_train, y_test):
    print("--- Step 4: Optunaによるハイパーパラメータ最適化 ---")

    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 50, 500),
            'max_depth': trial.suggest_int('max_depth', 3, 9),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'random_state': 42,
            'n_jobs': -1
        }

        model = xgb.XGBRegressor(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        return rmse

    print("Optunaによる最適化を開始します（約50回試行）...")
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=50, timeout=600)

    print("\n--- 最適化完了 ---")
    print(f"Best RMSE (Log scale): {study.best_value:.4f}")
    print("Best Params:", study.best_params)

    return study.best_params

#実行用
if __name__ == "__main__":
    # 1. Step 3 で保存した .npz データファイルを絶対パスで読み込む
    import_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\03_2_split_dataset.npz"
    data = np.load(import_path, allow_pickle=True)

    # 2. データを pandas の DataFrame / numpy 配列に復元する
    cols = data['columns'].tolist()
    X_train = pd.DataFrame(data['X_train'], columns=cols)
    X_test = pd.DataFrame(data['X_test'], columns=cols)
    y_train = data['y_train']
    y_test = data['y_test']

    # 3. 最適化関数を実行し、ベストパラメータを取得する
    best_params = optimize_hyperparameters(X_train, X_test, y_train, y_test)

    # 4. Step 5 でコピペしやすいように、パラメータをテキストファイルにも保存
    param_path = r"C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\04_2_best_params.txt"
    with open(param_path, "w") as f:
        f.write(str(best_params))

    print(f"\nベストパラメータをテキストとして保存しました：\n{param_path}")

--- Step 4: Optunaによるハイパーパラメータ最適化 ---
Optunaによる最適化を開始します（約50回試行）...

--- 最適化完了 ---
Best RMSE (Log scale): 0.3713
Best Params: {'n_estimators': 82, 'max_depth': 6, 'learning_rate': 0.011383782665697943, 'subsample': 0.745091768711668, 'colsample_bytree': 0.8564227395786601, 'min_child_weight': 3}

ベストパラメータをテキストとして保存しました：
C:\Users\scarl\Downloads\大学関連\25講義関連\b3後期\b4\coating_ML2\coating_ML_260618\data\interim\260625\wvp_adjustment\04_2_best_params.txt
